# Seminário 4: Fundamentos de Machine Learning**Disciplina:** Redes Neurais na Resolução de EDOs/EDPs — Doutorado PPGMC/UESC  **Referência:** Goodfellow, Bengio & Courville. *Deep Learning*, Cap. 5 — Machine Learning Basics (2016, MIT Press)  **Autor:** Guilherme de Sena Hughes---## Roteiro (~60 min)| # | Tópico | Tempo ||---|--------|-------|| 1 | O que é aprendizado de máquina? | ~3 min || 2 | Algoritmos de Aprendizado + Regressão Linear | ~7 min || 3 | Capacidade, Overfitting e Underfitting | ~7 min || 4 | Hiperparâmetros e Conjuntos de Validação | ~5 min || 5 | Estimadores, Viés e Variância | ~5 min || 6 | Estimação por Máxima Verossimilhança | ~5 min || 7 | Estatística Bayesiana e MAP | ~5 min || 8 | Algoritmos Supervisionados e Não Supervisionados | ~5 min || 9 | Gradiente Descendente Estocástico | ~5 min || 10 | Construindo um Algoritmo de ML | ~5 min || 11 | Desafios que Motivam Deep Learning | ~8 min |

In [ ]:
import torchimport torch.nn as nnimport torch.optim as optimimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.datasets import make_classification, make_blobsfrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScalerfrom sklearn.neighbors import KNeighborsClassifiertorch.manual_seed(42)np.random.seed(42)plt.rcParams['figure.figsize'] = (10, 5)plt.rcParams['font.size'] = 12print(f'PyTorch: {torch.__version__}')print(f'CUDA disponível: {torch.cuda.is_available()}')device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Dispositivo: {device}')

---## 1. O que é Aprendizado de Máquina? (~3 min)A definição clássica de Mitchell (1997), citada por Goodfellow:> *"Um programa de computador aprende com a experiência E em relação a uma classe de tarefas T e medida de desempenho P, se seu desempenho nas tarefas T, medido por P, melhora com a experiência E."*Em ML, ao contrário de programação tradicional, **não codificamos as regras explicitamente** — deixamos o algoritmo inferí-las a partir dos dados.```Programação tradicional:  Dados + Regras  →  RespostasMachine Learning:         Dados + Respostas →  Regras```

---## 2. Algoritmos de Aprendizado (~7 min)### Tarefa (T)O que o modelo deve fazer: classificação, regressão, geração, tradução, etc.### Medida de Desempenho (P)Como avaliamos o modelo: acurácia, MSE, log-likelihood, F1-score, etc.### Experiência (E)Os dados a partir dos quais o modelo aprende. Podem ser rotulados (supervisionados) ou não (não supervisionados).### O Conjunto de DadosUm dataset é tipicamente representado como uma **matriz de design** $X \in \mathbb{R}^{m \times n}$, onde:- $m$ = número de exemplos- $n$ = número de featuresCada exemplo $x^{(i)} \in \mathbb{R}^n$ é um vetor de features.

In [ ]:
# Criando um dataset simples de regressão# y = 2x + 1 + ruído gaussianom = 200  # exemplosX_np = np.linspace(-3, 3, m).reshape(-1, 1)y_np = 2 * X_np + 1 + np.random.randn(m, 1) * 0.8X = torch.tensor(X_np, dtype=torch.float32)y = torch.tensor(y_np, dtype=torch.float32)plt.scatter(X_np, y_np, alpha=0.5, s=15, label='Dados')plt.plot(X_np, 2*X_np + 1, 'r-', lw=2, label='Função real: y = 2x + 1')plt.xlabel('x')plt.ylabel('y')plt.title('Dataset de regressão com ruído gaussiano')plt.legend()plt.grid(True, alpha=0.3)plt.show()print(f'Shape de X: {X.shape}  →  {m} exemplos, 1 feature')print(f'Shape de y: {y.shape}')

### Exemplo Canônico: Regressão Linear (Seção 5.1.4)O livro apresenta a regressão linear como o primeiro algoritmo de aprendizado completo. O modelo assume:$$\hat{y} = X w \quad \text{onde} \quad w \in \mathbb{R}^n$$A função de custo é o erro quadrático médio (MSE):$$\mathcal{L}(w) = \frac{1}{m} \|\hat{y} - y\|_2^2 = \frac{1}{m} \|Xw - y\|_2^2$$Derivando e igualando a zero, obtemos a **solução em forma fechada** (equações normais):$$\nabla_w \mathcal{L} = 0 \implies w^* = (X^\top X)^{-1} X^\top y$$

In [ ]:
# Regressão Linear: solução analítica via equações normais# Adicionamos coluna de 1s para o bias (intercepto)X_bias = np.hstack([X_np, np.ones((m, 1))])  # [x, 1] → w = [slope, intercept]# Equações normais: w* = (X^T X)^{-1} X^T yw_star = np.linalg.solve(X_bias.T @ X_bias, X_bias.T @ y_np)print(f'Solução analítica (equações normais):')print(f'  w (inclinação) = {w_star[0, 0]:.4f}  (real: 2.0)')print(f'  b (intercepto)  = {w_star[1, 0]:.4f}  (real: 1.0)')# Visualizaçãoy_pred_normal = X_bias @ w_starplt.figure(figsize=(8, 5))plt.scatter(X_np, y_np, alpha=0.4, s=15, label='Dados')plt.plot(X_np, 2*X_np + 1, 'r-', lw=2, label='Função real: y = 2x + 1')plt.plot(X_np, y_pred_normal, 'g--', lw=2, label=f'Equações normais: y = {w_star[0,0]:.2f}x + {w_star[1,0]:.2f}')plt.xlabel('x')plt.ylabel('y')plt.title('Regressão Linear: Solução Analítica via Equações Normais')plt.legend()plt.grid(True, alpha=0.3)plt.show()mse = np.mean((y_pred_normal - y_np)**2)print(f'MSE no dataset: {mse:.4f}')

---## 3. Capacidade, Overfitting e Underfitting (~7 min)O objetivo central de ML é que o modelo **generalize** — ou seja, que tenha bom desempenho em dados **nunca vistos**.### Erro de Treinamento vs. Erro de Generalização- **Erro de treinamento** ($\mathcal{L}_{train}$): erro medido nos dados de treino- **Erro de generalização** ($\mathcal{L}_{test}$): erro esperado em novos dados (estimado pelo conjunto de teste)### Capacidade do Modelo**Capacidade** é a habilidade do modelo de se ajustar a uma grande variedade de funções. Modelos com pouca capacidade não conseguem capturar padrões complexos; modelos com capacidade excessiva memorizam os dados de treino.| Situação | Erro Treino | Erro Teste | Causa ||----------|-------------|------------|-------|| **Underfitting** | Alto | Alto | Capacidade insuficiente || **Ideal** | Baixo | Baixo | Capacidade adequada || **Overfitting** | Baixo | Alto | Capacidade excessiva |

In [ ]:
# Demonstração de underfitting vs. overfitting via regressão polinomialfrom numpy.polynomial import polynomial as PX_train_np = np.sort(np.random.uniform(-3, 3, 30))y_train_np = np.sin(X_train_np) + np.random.randn(30) * 0.3X_test_np = np.linspace(-3, 3, 200)y_true = np.sin(X_test_np)graus = [1, 4, 20]  # underfitting, ideal, overfittinglabels = ['Grau 1 (Underfitting)', 'Grau 4 (Bom fit)', 'Grau 20 (Overfitting)']cores = ['blue', 'green', 'red']fig, axes = plt.subplots(1, 3, figsize=(15, 4))for ax, grau, label, cor in zip(axes, graus, labels, cores):    coef = np.polyfit(X_train_np, y_train_np, grau)    y_pred_train = np.polyval(coef, X_train_np)    y_pred_test = np.polyval(coef, X_test_np)    train_mse = np.mean((y_train_np - y_pred_train)**2)    test_mse = np.mean((y_true - y_pred_test)**2)    ax.scatter(X_train_np, y_train_np, color='black', s=20, zorder=5, label='Treino')    ax.plot(X_test_np, y_true, 'k--', alpha=0.4, label='Função real')    ax.plot(X_test_np, np.clip(y_pred_test, -3, 3), color=cor, lw=2, label='Modelo')    ax.set_title(f'{label}\nMSE treino={train_mse:.3f} | MSE teste={test_mse:.3f}')    ax.set_ylim(-2.5, 2.5)    ax.legend(fontsize=8)    ax.grid(True, alpha=0.3)plt.suptitle('Capacidade do Modelo: Underfitting → Ideal → Overfitting', fontsize=13, y=1.02)plt.tight_layout()plt.show()

### RegularizaçãoPara controlar overfitting sem simplesmente reduzir a capacidade, usamos **regularização** — penalizamos a complexidade do modelo na função de perda:$$\mathcal{L}_{reg}(\theta) = \mathcal{L}(\theta) + \lambda \cdot \Omega(\theta)$$- **L2 (Ridge / Weight Decay):** $\Omega(\theta) = \|\theta\|_2^2$ — penaliza pesos grandes, encolhe-os em direção a zero- **L1 (Lasso):** $\Omega(\theta) = \|\theta\|_1$ — promove esparsidade (alguns pesos vão exatamente a zero)O hiperparâmetro $\lambda$ controla o trade-off entre ajuste aos dados e simplicidade do modelo.

In [ ]:
# Efeito da regularização L2 na regressão polinomial# Sem regularização, o polinômio de grau 20 explode (overfitting)# Com regularização, os coeficientes são controladosfrom numpy.polynomial import polynomial as Pgrau = 15X_train_reg = np.sort(np.random.uniform(-3, 3, 25))y_train_reg = np.sin(X_train_reg) + np.random.randn(25) * 0.3X_plot = np.linspace(-3, 3, 200)# Construir matriz de Vandermondedef vandermonde(x, d):    return np.vstack([x**i for i in range(d+1)]).TV_train = vandermonde(X_train_reg, grau)V_plot  = vandermonde(X_plot, grau)lambdas = [0, 1e-6, 1e-3, 1.0]labels_reg = ['λ=0 (sem reg.)', 'λ=1e-6', 'λ=1e-3', 'λ=1.0']cores_reg = ['red', 'orange', 'green', 'blue']fig, axes = plt.subplots(1, 4, figsize=(18, 4))for ax, lam, lab, cor in zip(axes, lambdas, labels_reg, cores_reg):    # Solução Ridge: w = (V^T V + λI)^{-1} V^T y    I = np.eye(grau + 1)    w = np.linalg.solve(V_train.T @ V_train + lam * I, V_train.T @ y_train_reg)    y_plot = V_plot @ w        ax.scatter(X_train_reg, y_train_reg, color='black', s=20, zorder=5)    ax.plot(X_plot, np.sin(X_plot), 'k--', alpha=0.4, label='sin(x)')    ax.plot(X_plot, np.clip(y_plot, -3, 3), color=cor, lw=2, label=lab)    ax.set_title(f'{lab}\n||w||₂ = {np.linalg.norm(w):.1f}')    ax.set_ylim(-2.5, 2.5)    ax.legend(fontsize=8)    ax.grid(True, alpha=0.3)plt.suptitle(f'Efeito da Regularização L2 (Ridge) em Polinômio de Grau {grau}', fontsize=13, y=1.02)plt.tight_layout()plt.show()

---## 4. Hiperparâmetros e Conjuntos de Validação (~5 min)### Parâmetros vs. Hiperparâmetros- **Parâmetros** ($\theta$): aprendidos durante o treinamento (pesos, biases)- **Hiperparâmetros**: definidos *antes* do treinamento (taxa de aprendizado, número de camadas, $\lambda$ de regularização, etc.)### Divisão dos Dados```Dataset completo│├── Treino   (~70%)  → ajusta os parâmetros θ├── Validação (~15%) → seleciona hiperparâmetros└── Teste    (~15%)  → avaliação final (usado UMA vez)```> ⚠️ **Regra de ouro:** o conjunto de teste nunca deve influenciar decisões de modelagem. Se você usa o teste para escolher hiperparâmetros, ele se torna validação — e você perde a estimativa honesta de generalização.### Cross-Validation (k-fold)Quando o dataset é pequeno, dividir em treino/validação desperdiça dados. A alternativa é **k-fold cross-validation**: divide os dados em $k$ partições, treina $k$ vezes usando $k-1$ partições para treino e 1 para validação, alternando.

In [ ]:
# Demonstração de divisão treino/validação/teste e busca de hiperparâmetroX_full, y_full = make_classification(    n_samples=1000, n_features=10, n_informative=5,    n_redundant=2, random_state=42)X_temp, X_test, y_temp, y_test = train_test_split(X_full, y_full, test_size=0.15, random_state=42)X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.18, random_state=42)print(f'Treino:    {X_train.shape[0]} exemplos ({100*X_train.shape[0]/1000:.0f}%)')print(f'Validação: {X_val.shape[0]} exemplos ({100*X_val.shape[0]/1000:.0f}%)')print(f'Teste:     {X_test.shape[0]} exemplos ({100*X_test.shape[0]/1000:.0f}%)')# Busca de hiperparâmetro: qual learning rate é melhor?scaler = StandardScaler()X_train_s = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)X_val_s   = torch.tensor(scaler.transform(X_val), dtype=torch.float32)y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)y_val_t   = torch.tensor(y_val,   dtype=torch.float32).unsqueeze(1)lrs = [0.1, 0.01, 0.001, 0.0001]val_losses = []for lr in lrs:    model = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())    opt = optim.SGD(model.parameters(), lr=lr)    loss_fn = nn.BCELoss()    for _ in range(200):        model.train()        opt.zero_grad()        loss = loss_fn(model(X_train_s), y_train_t)        loss.backward()        opt.step()    model.eval()    with torch.no_grad():        val_loss = loss_fn(model(X_val_s), y_val_t).item()    val_losses.append(val_loss)plt.figure(figsize=(7, 4))plt.bar([str(lr) for lr in lrs], val_losses, color=['#2196F3','#4CAF50','#FF9800','#F44336'])plt.xlabel('Learning Rate')plt.ylabel('Perda de Validação (BCE)')plt.title('Busca de Hiperparâmetro: Learning Rate × Perda de Validação')plt.grid(True, alpha=0.3, axis='y')melhor_lr = lrs[np.argmin(val_losses)]print(f'\nMelhor learning rate: {melhor_lr} (val loss = {min(val_losses):.4f})')plt.show()

---## 5. Estimadores, Viés e Variância (~5 min)### EstimadorUm **estimador** $\hat{\theta}_m$ é uma função dos dados que produz uma estimativa do parâmetro verdadeiro $\theta^*$:$$\hat{\theta}_m = g(x^{(1)}, x^{(2)}, \ldots, x^{(m)})$$### Viés$$\text{viés}(\hat{\theta}_m) = \mathbb{E}[\hat{\theta}_m] - \theta^*$$- **Estimador não-viesado:** $\mathbb{E}[\hat{\theta}_m] = \theta^*$- **Estimador assintoticamente não-viesado:** $\lim_{m\to\infty} \mathbb{E}[\hat{\theta}_m] = \theta^*$### Variância$$\text{Var}(\hat{\theta}) = \mathbb{E}[(\hat{\theta} - \mathbb{E}[\hat{\theta}])^2]$$### Decomposição do Erro Quadrático Médio$$\text{MSE} = \mathbb{E}[(\hat{\theta}_m - \theta^*)^2] = \text{viés}^2 + \text{Var}(\hat{\theta}_m)$$Este é o famoso **trade-off viés-variância**: aumentar a capacidade reduz o viés mas aumenta a variância.### Consistência (Seção 5.4.4)Um estimador é **consistente** se ele converge em probabilidade para o valor verdadeiro:$$\text{plim}_{m \to \infty} \hat{\theta}_m = \theta^*$$Consistência garante que, com dados suficientes, o estimador converge. É uma propriedade assintótica — um estimador viesado pode ser consistente se o viés desaparece com $m \to \infty$.

In [ ]:
# Ilustração visual do trade-off viés-variânciafig, axes = plt.subplots(1, 2, figsize=(12, 5))# Simulação: estimadores com diferentes viés e variânciatheta_true = 5.0n_amostras = 500n_experimentos = 200# Estimador A: baixo viés, alta variância (modelo complexo)est_A = theta_true + np.random.randn(n_experimentos) * 2.0# Estimador B: alto viés, baixa variância (modelo simples)est_B = theta_true + 1.5 + np.random.randn(n_experimentos) * 0.5for ax, est, label, cor in zip(axes,                                [est_A, est_B],                                ['A: Baixo Viés, Alta Variância', 'B: Alto Viés, Baixa Variância'],                                ['#2196F3', '#FF9800']):    vies = np.mean(est) - theta_true    variancia = np.var(est)    mse = vies**2 + variancia    ax.hist(est, bins=30, color=cor, alpha=0.7, edgecolor='white')    ax.axvline(theta_true, color='black', lw=2, ls='--', label=f'θ* = {theta_true}')    ax.axvline(np.mean(est), color=cor, lw=2, label=f'E[θ̂] = {np.mean(est):.2f}')    ax.set_title(f'{label}\nViés={vies:.2f} | Var={variancia:.2f} | MSE={mse:.2f}')    ax.set_xlabel('Estimativa θ̂')    ax.legend()    ax.grid(True, alpha=0.3)plt.suptitle('Trade-off Viés-Variância', fontsize=13)plt.tight_layout()plt.show()

In [ ]:
# Consistência: o estimador converge para θ* com m → ∞# Exemplo: variância amostral com denominador m (viesado, mas consistente)theta_true_var = 4.0  # variância real (σ² = 4, σ = 2)tamanhos_m = [5, 10, 30, 100, 300, 1000, 5000]n_repeticoes = 500# Estimador viesado da variância: (1/m) Σ(x - x̄)² → viés = -σ²/m# Estimador não-viesado: (1/(m-1)) Σ(x - x̄)²medias_viesado = []medias_nao_viesado = []for m_size in tamanhos_m:    ests_v = []    ests_nv = []    for _ in range(n_repeticoes):        amostra = np.random.normal(0, np.sqrt(theta_true_var), m_size)        ests_v.append(np.var(amostra, ddof=0))      # viés: divide por m        ests_nv.append(np.var(amostra, ddof=1))      # não-viesado: divide por m-1    medias_viesado.append(np.mean(ests_v))    medias_nao_viesado.append(np.mean(ests_nv))plt.figure(figsize=(10, 5))plt.plot(tamanhos_m, medias_viesado, 'o-', color='#F44336', lw=2, ms=8, label='Viesado (÷ m)')plt.plot(tamanhos_m, medias_nao_viesado, 's-', color='#4CAF50', lw=2, ms=8, label='Não-viesado (÷ m-1)')plt.axhline(theta_true_var, color='black', ls='--', lw=2, label=f'σ² real = {theta_true_var}')plt.xscale('log')plt.xlabel('Número de amostras (m)')plt.ylabel('E[σ̂²]')plt.title('Consistência: ambos estimadores convergem para σ² com m → ∞\n(o viesado tem viés = −σ²/m que desaparece assintoticamente)')plt.legend()plt.grid(True, alpha=0.3)plt.show()

---## 6. Estimação por Máxima Verossimilhança (MLE) (~5 min)A maioria dos algoritmos de ML pode ser interpretada como **maximização da verossimilhança** dos dados observados.Dado um dataset $\mathcal{D} = \{x^{(1)}, \ldots, x^{(m)}\}$ i.i.d. segundo $p_{dados}$, o estimador de máxima verossimilhança é:$$\theta_{ML} = \arg\max_{\theta} \prod_{i=1}^{m} p_{modelo}(x^{(i)}; \theta)$$Na prática, trabalhamos com o **log-likelihood** (mais estável numericamente):$$\theta_{ML} = \arg\max_{\theta} \sum_{i=1}^{m} \log p_{modelo}(x^{(i)}; \theta)$$### Conexão fundamental: MLE ↔ KL Divergence ↔ Cross-Entropy (Seção 5.5.1)O livro mostra que maximizar o log-likelihood é **equivalente** a minimizar a divergência KL entre a distribuição empírica $\hat{p}_{dados}$ e o modelo $p_{modelo}$:$$D_{KL}(\hat{p}_{dados} \| p_{modelo}) = \underbrace{\mathbb{E}_{x \sim \hat{p}}[\log \hat{p}(x)]}_{\text{entropia (constante)}} - \underbrace{\mathbb{E}_{x \sim \hat{p}}[\log p_{modelo}(x; \theta)]}_{\text{este é o neg. log-likelihood!}}$$Como o primeiro termo é constante (não depende de $\theta$), minimizar $D_{KL}$ é equivalente a maximizar o log-likelihood, que por sua vez é equivalente a **minimizar a cross-entropy**:$$H(\hat{p}, p_{modelo}) = -\mathbb{E}_{x \sim \hat{p}}[\log p_{modelo}(x; \theta)]$$> 💡 **Isso explica por que cross-entropy é a loss padrão em classificação:** ela é simplesmente o MLE para modelos de Bernoulli/Categorical!

In [ ]:
# MLE: estimando a média de uma Gaussiana# O MLE para a média de N(μ, σ²) com σ² conhecido é simplesmente a média amostralmu_true = 3.0sigma = 1.5tamanhos = [5, 20, 100, 1000]fig, axes = plt.subplots(1, 4, figsize=(16, 4))for ax, m in zip(axes, tamanhos):    amostras = np.random.normal(mu_true, sigma, m)    mu_mle = np.mean(amostras)  # MLE da média    x_range = np.linspace(mu_true - 5, mu_true + 5, 300)    p_true = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((x_range - mu_true)/sigma)**2)    p_mle  = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((x_range - mu_mle)/sigma)**2)    ax.hist(amostras, bins=min(20, m//2+2), density=True, alpha=0.4, color='steelblue', label='Amostras')    ax.plot(x_range, p_true, 'k-', lw=2, label=f'Real (μ={mu_true})')    ax.plot(x_range, p_mle, 'r--', lw=2, label=f'MLE (μ̂={mu_mle:.2f})')    ax.set_title(f'm = {m} amostras')    ax.legend(fontsize=7)    ax.grid(True, alpha=0.3)plt.suptitle('MLE da Média Gaussiana: Convergência com mais dados', fontsize=12)plt.tight_layout()plt.show()

In [ ]:
# Demonstração: MLE minimiza a KL divergence entre dados e modelo# Exemplo: estimando p de uma Bernoullifrom scipy.special import rel_entrp_real = 0.3  # probabilidade real de "sucesso"n_amostras_ce = 100np.random.seed(42)dados = np.random.binomial(1, p_real, n_amostras_ce)p_empirico = np.mean(dados)  # frequência observadap_modelo_range = np.linspace(0.01, 0.99, 200)# Negative log-likelihood (normalizado)nll = -(p_empirico * np.log(p_modelo_range) + (1 - p_empirico) * np.log(1 - p_modelo_range))# KL divergence: D_KL(p_emp || p_modelo)kl = p_empirico * np.log(p_empirico / p_modelo_range) + (1 - p_empirico) * np.log((1 - p_empirico) / (1 - p_modelo_range))fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].plot(p_modelo_range, nll, 'b-', lw=2)axes[0].axvline(p_empirico, color='red', ls='--', lw=2, label=f'MLE = p̂ = {p_empirico:.2f}')axes[0].axvline(p_real, color='black', ls=':', lw=2, label=f'p real = {p_real}')axes[0].set_xlabel('p_modelo')axes[0].set_ylabel('Negative Log-Likelihood')axes[0].set_title('NLL: mínimo no MLE')axes[0].legend()axes[0].grid(True, alpha=0.3)axes[1].plot(p_modelo_range, kl, 'r-', lw=2)axes[1].axvline(p_empirico, color='red', ls='--', lw=2, label=f'Min KL = p̂ = {p_empirico:.2f}')axes[1].axvline(p_real, color='black', ls=':', lw=2, label=f'p real = {p_real}')axes[1].set_xlabel('p_modelo')axes[1].set_ylabel('D_KL(p̂_dados || p_modelo)')axes[1].set_title('KL Divergence: mínimo idêntico ao NLL')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.suptitle('MLE ≡ Minimizar KL Divergence ≡ Minimizar Cross-Entropy', fontsize=13)plt.tight_layout()plt.show()print(f'p real = {p_real:.2f}')print(f'p empírico (MLE) = {p_empirico:.2f}')print(f'NLL no MLE = {-(p_empirico * np.log(p_empirico) + (1-p_empirico) * np.log(1-p_empirico)):.4f}')

---## 7. Estatística Bayesiana e MAP (~5 min)Na visão bayesiana, $\theta$ não é um valor fixo desconhecido, mas uma **variável aleatória** com distribuição de probabilidade.### Teorema de Bayes$$p(\theta \mid \mathcal{D}) = \frac{p(\mathcal{D} \mid \theta) \cdot p(\theta)}{p(\mathcal{D})}$$| Termo | Nome | Significado ||-------|------|-------------|| $p(\theta)$ | Prior | Crença sobre θ *antes* de ver os dados || $p(\mathcal{D} \mid \theta)$ | Verossimilhança | Probabilidade dos dados dado θ || $p(\theta \mid \mathcal{D})$ | Posterior | Crença *atualizada* após ver os dados || $p(\mathcal{D})$ | Evidência | Constante de normalização |### MAP: Maximum A Posteriori (Seção 5.6.1)$$\theta_{MAP} = \arg\max_\theta [\log p(\mathcal{D}|\theta) + \log p(\theta)]$$Se a prior é **gaussiana** $p(\theta) = \mathcal{N}(0, \frac{1}{\lambda}I)$, então:$$\log p(\theta) = -\frac{\lambda}{2} \|\theta\|_2^2 + \text{const.}$$Portanto, **MAP com prior gaussiana = MLE + regularização L2 (weight decay)!**Isso mostra que a regularização tem uma **interpretação bayesiana**: estamos incorporando uma crença a priori de que os pesos devem ser pequenos.

In [ ]:
# Atualização Bayesiana: estimando p (prob. de cara em uma moeda)# Prior: Beta(α, β) — conjugado para Bernoulli# Posterior: Beta(α + k, β + n - k)from scipy.stats import beta as beta_distp_true = 0.7  # moeda viciada — probabilidade real de carap_range = np.linspace(0.01, 0.99, 300)# Prior: uniforme Beta(1, 1) — sem conhecimento prévioalpha_prior, beta_prior = 1, 1sequencias = [0, 5, 20, 100]fig, axes = plt.subplots(1, 4, figsize=(16, 4))np.random.seed(42)lancamentos = np.random.binomial(1, p_true, max(sequencias))for ax, n in zip(axes, sequencias):    k = np.sum(lancamentos[:n]) if n > 0 else 0    alpha_post = alpha_prior + k    beta_post  = beta_prior + (n - k)    prior = beta_dist.pdf(p_range, alpha_prior, beta_prior)    posterior = beta_dist.pdf(p_range, alpha_post, beta_post)    ax.plot(p_range, prior, 'b--', lw=1.5, alpha=0.6, label='Prior')    ax.plot(p_range, posterior, 'r-', lw=2, label='Posterior')    ax.axvline(p_true, color='black', ls=':', lw=1.5, label=f'p real={p_true}')    if n > 0:        ax.axvline(k/n, color='green', ls='-.', lw=1.5, label=f'MLE={k/n:.2f}')    ax.set_title(f'n={n} lançamentos\n({k} caras)' if n > 0 else 'Antes dos dados')    ax.set_xlabel('p (prob. de cara)')    ax.legend(fontsize=7)    ax.set_ylim(0, None)    ax.grid(True, alpha=0.3)plt.suptitle('Atualização Bayesiana: Prior → Posterior com mais dados', fontsize=12)plt.tight_layout()plt.show()

In [ ]:
# MAP com prior gaussiana = MLE + Regularização L2# Demonstração: regressão com pesos grandes vs. MAP com priornp.random.seed(42)X_map = np.sort(np.random.uniform(-2, 2, 20))y_map = 0.5 * X_map + np.random.randn(20) * 0.5grau_map = 8V_map = np.vstack([X_map**i for i in range(grau_map+1)]).T# MLE puro: minimiza ||Vw - y||²w_mle = np.linalg.lstsq(V_map, y_map, rcond=None)[0]# MAP com prior gaussiana N(0, 1/λ): minimiza ||Vw - y||² + λ||w||²lambda_map = 0.1w_map = np.linalg.solve(V_map.T @ V_map + lambda_map * np.eye(grau_map+1), V_map.T @ y_map)X_plot_map = np.linspace(-2.2, 2.2, 200)V_plot_map = np.vstack([X_plot_map**i for i in range(grau_map+1)]).Tfig, axes = plt.subplots(1, 2, figsize=(14, 5))for ax, w, titulo in zip(axes, [w_mle, w_map],                           [f'MLE (sem prior)\n||w||₂ = {np.linalg.norm(w_mle):.1f}',                           f'MAP com prior gaussiana (λ={lambda_map})\n||w||₂ = {np.linalg.norm(w_map):.1f}']):    y_plot_map = V_plot_map @ w    ax.scatter(X_map, y_map, color='black', s=30, zorder=5, label='Dados')    ax.plot(X_plot_map, np.clip(y_plot_map, -3, 3), 'r-', lw=2, label='Modelo')    ax.plot(X_plot_map, 0.5 * X_plot_map, 'k--', alpha=0.4, label='Função real')    ax.set_ylim(-3, 3)    ax.set_title(titulo)    ax.legend(fontsize=9)    ax.grid(True, alpha=0.3)plt.suptitle('MAP com prior gaussiana ≡ Regularização L2 (Weight Decay)', fontsize=13)plt.tight_layout()plt.show()print(f'Norma dos pesos MLE: {np.linalg.norm(w_mle):.2f}')print(f'Norma dos pesos MAP: {np.linalg.norm(w_map):.2f}')print(f'→ A prior gaussiana "encolhe" os pesos, prevenindo overfitting')

---## 8. Algoritmos Supervisionados e Não Supervisionados (~5 min)### Supervisionado (Seção 5.7)O dataset contém pares $(x^{(i)}, y^{(i)})$. O objetivo é aprender $p(y|x)$ ou uma função $f: x \mapsto y$.- **Regressão:** $y \in \mathbb{R}$ (valor contínuo)- **Classificação:** $y \in \{1, \ldots, K\}$ (classe discreta)**Algoritmos clássicos discutidos no livro:**- **k-Nearest Neighbors (k-NN):** classifica pelo voto dos $k$ vizinhos mais próximos — baseado puramente em suavidade local- **Support Vector Machine (SVM):** encontra o hiperplano de máxima margem usando um kernel- **Árvores de Decisão:** particiona o espaço em regiões retangulares> 💡 Esses algoritmos dependem da suposição de **suavidade local** — funcionam bem em baixas dimensões, mas sofrem com a maldição da dimensionalidade (Seção 11).### Não Supervisionado (Seção 5.8)O dataset contém apenas $\{x^{(i)}\}$, sem rótulos. O objetivo é aprender a estrutura de $p(x)$.- **Clustering** (ex: K-Means): agrupa exemplos similares- **Redução de dimensionalidade** (ex: PCA): extrai representações compactas- **Modelos generativos** (ex: VAE, GAN): aprendem a gerar novos exemplos

In [ ]:
# Comparação de algoritmos supervisionados clássicos (Seção 5.7)from sklearn.neighbors import KNeighborsClassifierfrom sklearn.svm import SVCfrom sklearn.tree import DecisionTreeClassifierX_data, y_data = make_blobs(n_samples=300, centers=3, cluster_std=1.2, random_state=42)X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X_data, y_data, test_size=0.3, random_state=42)modelos_sk = {    'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5),    'SVM (RBF)': SVC(kernel='rbf', gamma='auto'),    'Árvore de Decisão': DecisionTreeClassifier(max_depth=5, random_state=42),}fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))h = 0.1x_min_s, x_max_s = X_data[:,0].min()-1, X_data[:,0].max()+1y_min_s, y_max_s = X_data[:,1].min()-1, X_data[:,1].max()+1xx_s, yy_s = np.meshgrid(np.arange(x_min_s, x_max_s, h), np.arange(y_min_s, y_max_s, h))for ax, (nome, clf) in zip(axes, modelos_sk.items()):    clf.fit(X_tr_s, y_tr_s)    acc_s = clf.score(X_te_s, y_te_s)    Z_s = clf.predict(np.c_[xx_s.ravel(), yy_s.ravel()]).reshape(xx_s.shape)        ax.contourf(xx_s, yy_s, Z_s, alpha=0.3, cmap='Set2')    ax.scatter(X_te_s[:,0], X_te_s[:,1], c=y_te_s, cmap='Set1', s=25, edgecolors='k', linewidth=0.5)    ax.set_title(f'{nome}\nAcurácia: {acc_s*100:.1f}%')    ax.grid(True, alpha=0.2)plt.suptitle('Algoritmos Supervisionados Clássicos (Seção 5.7)', fontsize=13, y=1.02)plt.tight_layout()plt.show()

In [ ]:
# Não Supervisionado: K-Means + PCAfrom sklearn.cluster import KMeansfrom sklearn.decomposition import PCAX_data_us, y_data_us = make_blobs(n_samples=300, centers=3, cluster_std=1.2, random_state=42)kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)y_cluster = kmeans.fit_predict(X_data_us)fig, axes = plt.subplots(1, 2, figsize=(12, 5))# Supervisionado: usa os rótulos reaisscatter1 = axes[0].scatter(X_data_us[:, 0], X_data_us[:, 1], c=y_data_us, cmap='Set1', s=30, alpha=0.8)axes[0].set_title('Supervisionado: Classes reais (y conhecidos)')axes[0].set_xlabel('Feature 1')axes[0].set_ylabel('Feature 2')axes[0].grid(True, alpha=0.3)# Não supervisionado: descobre clusters sem rótulosscatter2 = axes[1].scatter(X_data_us[:, 0], X_data_us[:, 1], c=y_cluster, cmap='Set2', s=30, alpha=0.8)axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],                marker='X', s=200, c='black', zorder=5, label='Centroides')axes[1].set_title('Não Supervisionado: K-Means (sem rótulos)')axes[1].set_xlabel('Feature 1')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.suptitle('Supervisionado vs. Não Supervisionado', fontsize=13)plt.tight_layout()plt.show()

---## 9. Gradiente Descendente Estocástico (SGD) (~5 min)A maioria dos algoritmos de deep learning usa **otimização baseada em gradiente**.### Gradiente Descendente (Batch)Atualiza os parâmetros usando o gradiente calculado sobre **todo o dataset**:$$\theta \leftarrow \theta - \eta \cdot \nabla_\theta \mathcal{L}(\theta; \mathcal{D})$$**Problema:** muito lento para datasets grandes.### SGD — Gradiente Descendente EstocásticoAtualiza os parâmetros usando o gradiente calculado sobre **um exemplo (ou mini-batch)**:$$\theta \leftarrow \theta - \eta \cdot \nabla_\theta \mathcal{L}(\theta; x^{(i)}, y^{(i)})$$| Variante | Exemplos por atualização | Característica ||----------|--------------------------|----------------|| Batch GD | Todos ($m$) | Estável, mas lento || SGD puro | 1 | Rápido, mas ruidoso || Mini-batch SGD | $b \in [32, 512]$ | Equilíbrio prático |O **ruído** do SGD tem efeito regularizador e ajuda a escapar de mínimos locais rasos.

In [ ]:
# Comparação: Batch GD vs. Mini-batch SGD vs. SGD puro# Tarefa: regressão linear simplesX_reg = torch.linspace(-2, 2, 500).unsqueeze(1)y_reg = 3 * X_reg + 1.5 + torch.randn_like(X_reg) * 0.5def treinar(X, y, batch_size, n_epochs=100, lr=0.05):    model = nn.Linear(1, 1)    nn.init.constant_(model.weight, 0.0)    nn.init.constant_(model.bias, 0.0)    opt = optim.SGD(model.parameters(), lr=lr)    loss_fn = nn.MSELoss()    historico = []    m = X.shape[0]    for epoch in range(n_epochs):        perm = torch.randperm(m)        X_s, y_s = X[perm], y[perm]        for i in range(0, m, batch_size):            Xb = X_s[i:i+batch_size]            yb = y_s[i:i+batch_size]            opt.zero_grad()            loss = loss_fn(model(Xb), yb)            loss.backward()            opt.step()        with torch.no_grad():            historico.append(loss_fn(model(X), y).item())    return historicoh_batch   = treinar(X_reg, y_reg, batch_size=500)  # Batch GDh_mini    = treinar(X_reg, y_reg, batch_size=32)   # Mini-batchh_sgd     = treinar(X_reg, y_reg, batch_size=1)    # SGD puroplt.figure(figsize=(10, 5))plt.plot(h_batch, label='Batch GD (b=500)', lw=2)plt.plot(h_mini,  label='Mini-batch SGD (b=32)', lw=2)plt.plot(h_sgd,   label='SGD puro (b=1)', lw=1.5, alpha=0.7)plt.xlabel('Época')plt.ylabel('MSE (dataset completo)')plt.title('Convergência: Batch GD vs. Mini-batch vs. SGD puro')plt.legend()plt.yscale('log')plt.grid(True, alpha=0.3)plt.show()

---## 10. Construindo um Algoritmo de Aprendizado de Máquina (~5 min)Goodfellow aponta que praticamente todos os algoritmos de deep learning combinam:1. **Uma especificação de dataset** → Seção 22. **Uma função de custo** (loss) → Seções 5–6 (viés-variância, MLE, cross-entropy)3. **Um procedimento de otimização** → Seção 9 (SGD)4. **Um modelo** → Seção 3 (capacidade) + Seção 8 (tipos de algoritmos)A **regularização** (Seção 3) e a perspectiva **bayesiana** (Seção 7) modificam a loss, enquanto os **hiperparâmetros** (Seção 4) controlam todos os componentes.### Exemplo completo: Classificação Binária com MLP em PyTorchVamos construir do zero um pipeline completo, seguindo exatamente essa estrutura.

In [ ]:
# ============================================================# Pipeline completo de ML em PyTorch# ============================================================# --- 1. DATASET ---X_np, y_np = make_classification(    n_samples=800, n_features=2, n_informative=2,    n_redundant=0, n_clusters_per_class=1, random_state=42)scaler = StandardScaler()X_np = scaler.fit_transform(X_np)X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=0.2, random_state=42)X_train_t = torch.tensor(X_tr, dtype=torch.float32)y_train_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)X_test_t  = torch.tensor(X_te, dtype=torch.float32)y_test_t  = torch.tensor(y_te, dtype=torch.float32).unsqueeze(1)# --- 2. MODELO ---class MLP(nn.Module):    def __init__(self):        super().__init__()        self.rede = nn.Sequential(            nn.Linear(2, 32),            nn.ReLU(),            nn.Linear(32, 16),            nn.ReLU(),            nn.Linear(16, 1),            nn.Sigmoid()        )    def forward(self, x):        return self.rede(x)modelo = MLP()print(modelo)total_params = sum(p.numel() for p in modelo.parameters())print(f'\nTotal de parâmetros: {total_params}')

In [ ]:
# --- 3. FUNÇÃO DE CUSTO + OTIMIZAÇÃO ---loss_fn   = nn.BCELoss()                          # Binary Cross-Entropy (≡ MLE para Bernoulli)otimizador = optim.Adam(modelo.parameters(), lr=0.01)  # Adam: SGD com momentum adaptativo# --- 4. LOOP DE TREINAMENTO ---N_EPOCHS  = 200BATCH     = 32hist_train, hist_test = [], []for epoch in range(N_EPOCHS):    modelo.train()    perm = torch.randperm(X_train_t.shape[0])    for i in range(0, X_train_t.shape[0], BATCH):        idx = perm[i:i+BATCH]        Xb, yb = X_train_t[idx], y_train_t[idx]        otimizador.zero_grad()        pred = modelo(Xb)        loss = loss_fn(pred, yb)        loss.backward()        otimizador.step()    modelo.eval()    with torch.no_grad():        loss_tr = loss_fn(modelo(X_train_t), y_train_t).item()        loss_te = loss_fn(modelo(X_test_t),  y_test_t ).item()        hist_train.append(loss_tr)        hist_test.append(loss_te)# Acurácia finalwith torch.no_grad():    pred_te = (modelo(X_test_t) >= 0.5).float()    acc = (pred_te == y_test_t).float().mean().item()print(f'Acurácia no teste: {acc*100:.1f}%')# Curvas de aprendizadoplt.figure(figsize=(10, 4))plt.plot(hist_train, label='Perda Treino')plt.plot(hist_test,  label='Perda Teste')plt.xlabel('Época')plt.ylabel('BCE Loss')plt.title('Curvas de Aprendizado')plt.legend()plt.grid(True, alpha=0.3)plt.show()

In [ ]:
# Fronteira de decisão aprendidah = 0.05x_min, x_max = X_np[:,0].min()-0.5, X_np[:,0].max()+0.5y_min, y_max = X_np[:,1].min()-0.5, X_np[:,1].max()+0.5xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))grade = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)modelo.eval()with torch.no_grad():    Z = modelo(grade).numpy().reshape(xx.shape)plt.figure(figsize=(8, 6))plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu', levels=50)plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)plt.scatter(X_te[:,0], X_te[:,1], c=y_te, cmap='RdBu', edgecolors='k', s=40, alpha=0.8)plt.title(f'Fronteira de Decisão — MLP (Acurácia: {acc*100:.1f}%)')plt.xlabel('Feature 1')plt.ylabel('Feature 2')plt.colorbar(label='P(classe=1)')plt.grid(True, alpha=0.2)plt.show()

---## 11. Desafios que Motivam Deep Learning (~8 min)A Seção 5.11 é a **ponte conceitual** entre ML clássico e Deep Learning. Os algoritmos tradicionais funcionam bem em muitos problemas, mas falham nas tarefas centrais de IA (reconhecimento de fala, visão, linguagem). O livro identifica **três desafios** fundamentais:### 11.1 A Maldição da DimensionalidadeO número de configurações possíveis cresce **exponencialmente** com o número de dimensões. Se distinguimos $v$ valores por dimensão em $d$ dimensões, precisamos de $O(v^d)$ exemplos para cobrir o espaço.> Em 1D com 10 regiões: 10 exemplos bastam. Em 10D: precisamos de $10^{10}$ exemplos!### 11.2 Suavidade Local e suas LimitaçõesAlgoritmos como k-NN, kernel machines e árvores de decisão dependem da hipótese de **suavidade local**: $f(x) \approx f(x + \epsilon)$ para $\epsilon$ pequeno. Cada exemplo só informa sobre sua vizinhança imediata, resultando em $O(k)$ regiões para $O(k)$ exemplos.Deep Learning supera isso: com $O(k)$ parâmetros, uma rede profunda pode representar $O(2^k)$ regiões, porque **introduz dependências entre regiões** via composição hierárquica de features.### 11.3 Manifold LearningA **hipótese do manifold**: dados reais (imagens, áudio, texto) se concentram em **variedades de baixa dimensão** imersos em espaços de alta dimensão. Imagens aleatórias são ruído — imagens "reais" ocupam uma fração ínfima do espaço de pixels.Se os dados vivem em um manifold de dimensão $d_{manifold} \ll d_{total}$, precisamos de muito menos exemplos do que o espaço total sugeriria.

In [ ]:
# Maldição da Dimensionalidade: volume da hipersfera colapsafrom scipy.special import gamma as gamma_fndef volume_hipersfera(d, r=1.0):    """Volume de uma hipersfera de raio r em d dimensões."""    return (np.pi**(d/2) / gamma_fn(d/2 + 1)) * r**ddef volume_hipercubo(d, lado=2.0):    """Volume de um hipercubo de lado 'lado' em d dimensões."""    return lado**ddims = np.arange(1, 51)razoes = [volume_hipersfera(d) / volume_hipercubo(d) for d in dims]fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Razão volume esfera/cuboaxes[0].plot(dims, razoes, 'b-', lw=2)axes[0].set_xlabel('Número de dimensões')axes[0].set_ylabel('V(esfera) / V(cubo)')axes[0].set_title('A hipersfera "desaparece" dentro do hipercubo\nFração do volume ocupado pela esfera inscrita')axes[0].set_yscale('log')axes[0].grid(True, alpha=0.3)axes[0].axhline(0.01, color='red', ls='--', alpha=0.5, label='1% do volume')axes[0].legend()# Distância média entre pontos uniformes cresce com dn_pontos = 1000dims_teste = [2, 5, 10, 50, 100, 500]dist_medias = []for d in dims_teste:    pontos = np.random.uniform(0, 1, (n_pontos, d))    # Distâncias entre pares aleatórios    idx = np.random.choice(n_pontos, (500, 2), replace=True)    dists = np.linalg.norm(pontos[idx[:,0]] - pontos[idx[:,1]], axis=1)    dist_medias.append((np.mean(dists), np.std(dists)))medias = [x[0] for x in dist_medias]stds = [x[1] for x in dist_medias]axes[1].errorbar(dims_teste, medias, yerr=stds, fmt='o-', color='#F44336', lw=2, capsize=5, ms=8)axes[1].set_xlabel('Número de dimensões')axes[1].set_ylabel('Distância média entre pontos')axes[1].set_title('Todos os pontos ficam "igualmente distantes"\n→ vizinhança local perde sentido')axes[1].set_xscale('log')axes[1].grid(True, alpha=0.3)plt.suptitle('A Maldição da Dimensionalidade (Seção 5.11.1)', fontsize=13, y=1.02)plt.tight_layout()plt.show()print('Em alta dimensão:')print(f'  - A esfera inscrita ocupa {razoes[-1]*100:.2e}% do cubo em {dims[-1]}D')print(f'  - Distância média entre pontos em 500D: {medias[-1]:.1f} ± {stds[-1]:.1f} (pontos se "espalham")')

In [ ]:
# k-NN falha com alta dimensionalidade (suavidade local não basta)# Teste: mesmo problema, embedding em dimensões crescentes com features de ruídofrom sklearn.neighbors import KNeighborsClassifiernp.random.seed(42)# Dataset base: 2 features informativasn_base = 500X_base, y_base = make_classification(n_samples=n_base, n_features=2, n_informative=2,                                      n_redundant=0, n_clusters_per_class=1, random_state=42)dims_ruido = [0, 5, 20, 50, 100, 200, 500]acc_knn = []acc_mlp_dim = []for d_noise in dims_ruido:    # Adiciona d_noise features de ruído    X_ruido = np.hstack([X_base, np.random.randn(n_base, d_noise)]) if d_noise > 0 else X_base.copy()        X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_ruido, y_base, test_size=0.3, random_state=42)        # Normalizar    sc_d = StandardScaler()    X_tr_d = sc_d.fit_transform(X_tr_d)    X_te_d = sc_d.transform(X_te_d)        # k-NN    knn = KNeighborsClassifier(n_neighbors=5)    knn.fit(X_tr_d, y_tr_d)    acc_knn.append(knn.score(X_te_d, y_te_d))        # MLP simples (2 camadas)    d_total = 2 + d_noise    mlp_d = nn.Sequential(        nn.Linear(d_total, 32), nn.ReLU(),        nn.Linear(32, 16), nn.ReLU(),        nn.Linear(16, 1), nn.Sigmoid()    )    opt_d = optim.Adam(mlp_d.parameters(), lr=0.01)    loss_d = nn.BCELoss()        X_tr_t_d = torch.tensor(X_tr_d, dtype=torch.float32)    y_tr_t_d = torch.tensor(y_tr_d, dtype=torch.float32).unsqueeze(1)    X_te_t_d = torch.tensor(X_te_d, dtype=torch.float32)        for _ in range(150):        mlp_d.train()        opt_d.zero_grad()        loss_d(mlp_d(X_tr_t_d), y_tr_t_d).backward()        opt_d.step()        mlp_d.eval()    with torch.no_grad():        pred_d = (mlp_d(X_te_t_d) >= 0.5).float().numpy().flatten()    acc_mlp_dim.append(np.mean(pred_d == y_te_d))total_dims = [2 + d for d in dims_ruido]plt.figure(figsize=(10, 5))plt.plot(total_dims, acc_knn, 'o-', color='#F44336', lw=2, ms=8, label='k-NN (k=5)')plt.plot(total_dims, acc_mlp_dim, 's-', color='#4CAF50', lw=2, ms=8, label='MLP (2 camadas)')plt.xlabel('Dimensão total (2 informativas + ruído)')plt.ylabel('Acurácia no teste')plt.title('k-NN vs. MLP: Degradação com dimensionalidade crescente\n(a rede neural é mais robusta a features irrelevantes)')plt.legend(fontsize=12)plt.grid(True, alpha=0.3)plt.xscale('log')plt.ylim(0.4, 1.05)plt.show()print('k-NN depende de distância euclidiana → features de ruído destroem a vizinhança')print('MLP aprende a ignorar features irrelevantes via pesos → mais robusta')

In [ ]:
# Manifold Learning: dados de alta dimensão vivem em variedades de baixa dimensãofrom sklearn.decomposition import PCAnp.random.seed(42)# Simular dados em um manifold: curva (1D) imersa em 3Dt = np.linspace(0, 4*np.pi, 500)X_manifold_3d = np.column_stack([    np.cos(t) + np.random.randn(500)*0.05,    np.sin(t) + np.random.randn(500)*0.05,    t / (4*np.pi) + np.random.randn(500)*0.05])fig = plt.figure(figsize=(16, 5))# Plot 3D do manifoldax1 = fig.add_subplot(131, projection='3d')ax1.scatter(X_manifold_3d[:,0], X_manifold_3d[:,1], X_manifold_3d[:,2],             c=t, cmap='viridis', s=5, alpha=0.7)ax1.set_title('Dados em 3D\n(vivem em manifold 1D)')ax1.set_xlabel('x₁')ax1.set_ylabel('x₂')ax1.set_zlabel('x₃')# PCA: variância explicada mostra dimensionalidade intrínsecapca = PCA().fit(X_manifold_3d)ax2 = fig.add_subplot(132)ax2.bar(range(1, 4), pca.explained_variance_ratio_, color=['#4CAF50', '#FF9800', '#F44336'])ax2.set_xlabel('Componente Principal')ax2.set_ylabel('Variância explicada')ax2.set_title('PCA: dimensionalidade intrínseca\n(maior parte da variância em 1-2 dims)')ax2.set_xticks([1, 2, 3])ax2.grid(True, alpha=0.3)# Imagens aleatórias vs. "reais" — a hipótese do manifoldax3 = fig.add_subplot(133)# Gerar "imagens" aleatórias de 8x8imgs_random = np.random.uniform(0, 1, (4, 8, 8))# Gerar "imagens" estruturadas (gradientes simples — simulam manifold)imgs_struct = []for i in range(4):    angle = np.pi * i / 4    x_g = np.linspace(-1, 1, 8)    y_g = np.linspace(-1, 1, 8)    xx_g, yy_g = np.meshgrid(x_g, y_g)    img = 0.5 + 0.5 * np.sin(xx_g * np.cos(angle) + yy_g * np.sin(angle))    imgs_struct.append(img)for i in range(4):    # Random    ax_sub1 = fig.add_axes([0.69 + i*0.075, 0.55, 0.065, 0.3])    ax_sub1.imshow(imgs_random[i], cmap='gray', vmin=0, vmax=1)    ax_sub1.axis('off')    if i == 0: ax_sub1.set_title('Aleatórias', fontsize=8, loc='left')        # Structured    ax_sub2 = fig.add_axes([0.69 + i*0.075, 0.12, 0.065, 0.3])    ax_sub2.imshow(imgs_struct[i], cmap='gray', vmin=0, vmax=1)    ax_sub2.axis('off')    if i == 0: ax_sub2.set_title('Estruturadas', fontsize=8, loc='left')ax3.axis('off')ax3.set_title('Hipótese do Manifold:\nimagens reais ≠ ruído', y=1.05)plt.suptitle('Manifold Learning: dados reais vivem em variedades de baixa dimensão (Seção 5.11.3)',              fontsize=13, y=1.05)plt.tight_layout()plt.show()print('A hipótese do manifold explica por que deep learning funciona:')print('→ Não precisamos cobrir todo o espaço R^n')print('→ Basta aprender a estrutura do manifold de baixa dimensão')print('→ Redes profundas aprendem representações hierárquicas desses manifolds')

---## Resumo| Conceito | Ponto-chave ||----------|-------------|| **Aprendizado** | Melhorar desempenho (P) em tarefa (T) com experiência (E) || **Regressão Linear** | Primeiro algoritmo completo: solução fechada via equações normais || **Capacidade** | Equilíbrio entre under e overfitting é o desafio central || **Regularização** | L2 controla norma dos pesos; L1 promove esparsidade || **Conjuntos** | Treino → parâmetros \| Validação → hiperparâmetros \| Teste → avaliação final || **Viés-Variância** | MSE = Viés² + Variância — trade-off inevitável || **Consistência** | Estimador converge para θ* com m → ∞ || **MLE** | Maximizar log-likelihood ≡ minimizar KL divergence ≡ minimizar cross-entropy || **MAP/Bayes** | MAP com prior gaussiana ≡ MLE + regularização L2 || **Algoritmos clássicos** | k-NN, SVM, árvores: dependem de suavidade local || **SGD** | Mini-batches: equilíbrio entre velocidade e estabilidade || **Pipeline** | Dataset + Modelo + Loss + Otimizador || **Maldição da dimensionalidade** | Espaço cresce exponencialmente → exemplos ficam esparsos || **Manifold Learning** | Dados reais vivem em variedades de baixa dimensão || **Deep Learning** | Supera suavidade local via composição hierárquica de features |### Conexão com PINNsEm **Physics-Informed Neural Networks** (tema da disciplina), todos esses conceitos se aplicam diretamente:- O **modelo** é uma rede que aproxima a solução de uma EDO/EDP- A **função de custo** inclui tanto resíduo da equação diferencial quanto condições de contorno → **MLE** do resíduo + **regularização** pela física- O **otimizador** (Adam/SGD) minimiza esse custo composto- **Overfitting** pode ocorrer quando a rede memoriza pontos de treinamento mas viola a física- A **hipótese do manifold** é especialmente relevante: soluções de EDPs vivem em variedades de baixa dimensão no espaço de funções- Deep Learning supera a **maldição da dimensionalidade** que limita métodos numéricos clássicos (FEM, FDM) em alta dimensão---### Referências- Goodfellow, I., Bengio, Y., Courville, A. (2016). *Deep Learning*, Cap. 5. MIT Press.- PyTorch Documentation: https://pytorch.org/docs/stable/- Raissi, M. et al. (2019). *Physics-informed neural networks.* Journal of Computational Physics.- Mitchell, T. (1997). *Machine Learning.* McGraw-Hill.